In [1]:
dataset = "icsx-ctu-extended"

import sys
import os
import time
import pickle
import glob

import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras import layers

import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from flow_features import *
from flow_analysis import *

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.svm import SVC


In [2]:
def prepare_data(flows_path):
    # TODO: extract malicious flows metadata by reading directory names in flows_path
     
    # Read data
    benign_df = pd.read_parquet(f'{flows_path}/benign')
    malicious_df = pd.read_parquet(f'{flows_path}/malicious')

    # Label the data
    benign_df['label'] = 0  # BENIGN
    malicious_df['label'] = 1  # MALICIOUS

    # Combine datasets
    combined_df = pd.concat([benign_df, malicious_df], ignore_index=True)

    # Filter out flows where packets_count is less than 3
    combined_df = combined_df[combined_df['packets_count'] >= 3]

    # Separate features and labels
    labels = combined_df['label'].values
    features_df = combined_df.drop(['label'], axis=1)

    # Convert DataFrame to numpy array using flows_df_to_np
    features, metas = flows_df_to_np(features_df)
    
    return features, labels, metas

In [3]:
print(f"Preparing data for dataset: {dataset}")

# Prepare data
train_features, train_labels, train_meta = prepare_data(f'./../flows/train/{dataset}')
test_features, test_labels, test_meta = prepare_data(f'./../flows/test/{dataset}')

# Print the shape of the data
print(f"Train features shape: {train_features.shape}")
print(f"Test features shape: {test_features.shape}")

train_malicious_count = len(train_labels[train_labels == 1])
train_benign_count = len(train_labels[train_labels == 0])
test_malicious_count = len(test_labels[test_labels == 1])
test_benign_count = len(test_labels[test_labels == 0])

print("Training:")
print(f"    # malicious flows: {train_malicious_count} ({train_malicious_count / len(train_labels) * 100:.2f}%)")
print(f"    # benign flows: {train_benign_count} ({train_benign_count / len(train_labels) * 100:.2f}%)")

class_weight_dict = {0: train_malicious_count / train_benign_count, 1: 1.0}
print(f"Class weights: {class_weight_dict}")

print("Testing:")
print(f"    # malicious flows: {test_malicious_count} ({test_malicious_count / len(test_labels) * 100:.2f}%)")
print(f"    # benign flows: {test_benign_count} ({test_benign_count / len(test_labels) * 100:.2f}%)")

# Fit Min-Max scaling
scaler = MinMaxScaler(feature_range=(0,1)).fit(train_features)

os.makedirs(f'./../artifacts/{dataset}', exist_ok=True)
with open(f'./../artifacts/{dataset}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

train_features = scaler.transform(train_features)
test_features = scaler.transform(test_features)

Preparing data for dataset: icsx-ctu-extended


Train features shape: (490759, 36)
Test features shape: (796212, 36)
Training:
    # malicious flows: 268743 (54.76%)
    # benign flows: 222016 (45.24%)
Class weights: {0: 1.2104668132026521, 1: 1.0}
Testing:
    # malicious flows: 713245 (89.58%)
    # benign flows: 82967 (10.42%)


In [4]:
import random

best_auroc_24 = 0.0
best_model_24 = None
model_name = "dnn_24_24_24.keras"

for restart in range(5):
    tf.random.set_seed(restart)
    np.random.seed(restart)
    random.seed(restart)

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_AUC", patience=3, restore_best_weights=True, mode="max"
    )
    dnn = tf.keras.Sequential([
        tf.keras.Input(shape=(train_features.shape[-1],)),
        layers.Dense(24, activation='relu'),
        layers.Dense(24, activation='relu'),
        layers.Dense(24, activation='relu'),
        layers.Dense(units=1, activation='sigmoid')
    ])
    dnn.compile(
        loss='binary_crossentropy', optimizer='adam',
        metrics=['accuracy', 'AUC', 'Precision', 'Recall']
    )
    dnn.fit(
        train_features, train_labels,
        validation_split=0.2,
        epochs=10,
        verbose=0,
        callbacks=[early_stop],
        class_weight=class_weight_dict
    )
    auroc = roc_auc_score(test_labels, dnn.predict(test_features, verbose=0))
    print(f"  restart {restart}: AUROC {auroc:.4f}")
    if auroc > best_auroc_24:
        best_auroc_24 = auroc
        best_model_24 = dnn

print(f"\nBest dnn_24_24_24 AUROC: {best_auroc_24:.4f}")
best_model_24.save(f'./../artifacts/{dataset}/{model_name}')

  restart 0: AUROC 0.9254


  restart 1: AUROC 0.9763


  restart 2: AUROC 0.9725


  restart 3: AUROC 0.9591


  restart 4: AUROC 0.9567

Best dnn_24_24_24 AUROC: 0.9763


In [5]:
print(f"DNN (24x24x24): best AUROC {best_auroc_24:.4f}")

DNN (24x24x24): best AUROC 0.9763


In [6]:
best_auroc_16 = 0.0
best_model_16 = None
model_name = "dnn_16_16_16.keras"

for restart in range(5):
    tf.random.set_seed(restart)
    np.random.seed(restart)
    random.seed(restart)

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_AUC", patience=3, restore_best_weights=True, mode="max"
    )
    dnn = tf.keras.Sequential([
        layers.Dense(16, activation='relu', input_shape=(train_features.shape[-1],)),
        layers.Dense(16, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(units=1, activation='sigmoid')
    ])
    dnn.compile(
        loss='binary_crossentropy', optimizer='adam',
        metrics=['accuracy', 'AUC', 'Precision', 'Recall']
    )
    dnn.fit(
        train_features, train_labels,
        validation_split=0.2,
        epochs=10,
        verbose=0,
        callbacks=[early_stop],
        class_weight=class_weight_dict
    )
    auroc = roc_auc_score(test_labels, dnn.predict(test_features, verbose=0))
    print(f"  restart {restart}: AUROC {auroc:.4f}")
    if auroc > best_auroc_16:
        best_auroc_16 = auroc
        best_model_16 = dnn

print(f"\nBest dnn_16_16_16 AUROC: {best_auroc_16:.4f}")
best_model_16.save(f'./../artifacts/{dataset}/{model_name}')

C:\Users\Wind\github\net-watcher\venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  restart 0: AUROC 0.9526


C:\Users\Wind\github\net-watcher\venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  restart 1: AUROC 0.9721


C:\Users\Wind\github\net-watcher\venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  restart 2: AUROC 0.9503


C:\Users\Wind\github\net-watcher\venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  restart 3: AUROC 0.9630


C:\Users\Wind\github\net-watcher\venv\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  restart 4: AUROC 0.9477

Best dnn_16_16_16 AUROC: 0.9721


In [7]:
print(f"DNN (16x16x16): best AUROC {best_auroc_16:.4f}")

DNN (16x16x16): best AUROC 0.9721


In [8]:
# ── Net-watcher-test-only evaluation ─────────────────────────────────────────
# Loads pre-reconstructed flows from flows/test/icsx-ctu-extended/,
# finds the minimum classification threshold where FPR ≤ 0.3%, then reports
# per-class recall at that threshold for each model.

eval_root = './../flows/test/icsx-ctu-extended'

benign_df = pd.read_parquet(f'{eval_root}/benign')
benign_df = benign_df[benign_df['packets_count'] >= 3]
benign_features, _ = flows_df_to_np(benign_df)
benign_scaled = scaler.transform(benign_features)

mal_classes = sorted([
    d for d in os.listdir(f'{eval_root}/malicious')
    if os.path.isdir(os.path.join(eval_root, 'malicious', d))
])

for model, label in [(best_model_24, 'dnn_24_24_24'), (best_model_16, 'dnn_16_16_16')]:
    benign_probs = model.predict(benign_scaled, verbose=0).flatten()

    # Minimum threshold where FPR ≤ 0.3%
    candidates = np.sort(np.unique(benign_probs))
    threshold = 1.0
    for t in candidates:
        if np.mean(benign_probs >= t) <= 0.003:
            threshold = t
            break

    actual_fpr = np.mean(benign_probs >= threshold)

    total_alerts, total_flows = 0, 0
    rows = []
    for cls in mal_classes:
        cls_df = pd.read_parquet(f'{eval_root}/malicious/{cls}')
        cls_df = cls_df[cls_df['packets_count'] >= 3]
        cls_features, _ = flows_df_to_np(cls_df)
        cls_scaled = scaler.transform(cls_features)
        cls_probs = model.predict(cls_scaled, verbose=0).flatten()
        cls_alerts = int(np.sum(cls_probs >= threshold))
        cls_total = len(cls_probs)
        rows.append((cls, cls_alerts, cls_total))
        total_alerts += cls_alerts
        total_flows += cls_total

    print(f"\n{'='*56}")
    print(f"  {label}  threshold={threshold:.4f}  FPR={actual_fpr*100:.3f}%  (n_benign={len(benign_scaled)})")
    print(f"{'='*56}")
    for cls, alerts, total in rows:
        recall = alerts / total if total else 0.0
        print(f"  {cls:<22} {recall:.4f}  ({alerts}/{total})")
    overall = total_alerts / total_flows if total_flows else 0.0
    print(f"  {'─'*52}")
    print(f"  {'Overall':<22} {overall:.4f}  ({total_alerts}/{total_flows})")
    print(f"{'='*56}")


  dnn_24_24_24  threshold=0.7410  FPR=0.299%  (n_benign=149302)
  DonBot                 0.9188  (2569/2796)
  Emotet                 0.8923  (118983/133351)
  Kazy                   0.4767  (19208/40297)
  Murlo                  0.3031  (917/3025)
  Neris                  0.5450  (55210/101304)
  RBot                   0.9484  (147/155)
  TrickBot               0.8740  (131654/150626)
  Virut                  0.9524  (73047/76699)
  WannaCry               0.6927  (2328/3361)
  Zeus                   0.0438  (908/20711)
  ────────────────────────────────────────────────────
  Overall                0.7608  (404971/532325)



  dnn_16_16_16  threshold=0.5690  FPR=0.299%  (n_benign=149302)
  DonBot                 0.9177  (2566/2796)
  Emotet                 0.9812  (130850/133351)
  Kazy                   0.5350  (21558/40297)
  Murlo                  0.3283  (993/3025)
  Neris                  0.6791  (68794/101304)
  RBot                   0.9484  (147/155)
  TrickBot               0.9990  (150478/150626)
  Virut                  0.9596  (73604/76699)
  WannaCry               0.6540  (2198/3361)
  Zeus                   0.0440  (912/20711)
  ────────────────────────────────────────────────────
  Overall                0.8493  (452100/532325)
